# Diagnostics

Checks used while developing the cleaning rules in `02_creating_table.ipynb`. They need the tables built there, and `df_net_clean` from its last cell for the spike diagnostic.

In [ ]:
import pyodbc
import pandas as pd

cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)
schema = 'lab_prj_emir_ecb'
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

In [ ]:
# diagnostics: which counterparty pairs drive the net position of a country on a given day?
def top_pairs(country, day, n=15):
    return pd.read_sql_query(f"""
    SELECT buyer_id, buyer_sector, seller_id, seller_sector,
           COUNT(*) AS n_rows, COUNT(DISTINCT product_id) AS n_contracts,
           SUM(notional) / 1e9 AS notional_bn,
           ROUND(AVG(notional), 0) AS avg_notional,
           ROUND(AVG(price), 2) AS avg_price,
           ROUND(AVG(quantity), 0) AS avg_quantity,
           ROUND(AVG(notional / NULLIF(quantity, 0)), 0) AS avg_notional_per_unit
    FROM {schema}.hermesf_fut
    WHERE product_country = '{country}' AND reference_period = '{day}'
    GROUP BY buyer_id, buyer_sector, seller_id, seller_sector
    ORDER BY notional_bn DESC
    LIMIT {n}
    """, cnxn)

# peaks seen in the charts
top_pairs('DE', '2019-05-31')

In [ ]:
top_pairs('US', '2021-06-30')

In [ ]:
top_pairs('ES', '2024-06-14')

In [ ]:
# diagnostics 2a: scale of notional relative to quantity x price, by country and year
# correct reports sit at 3 (quantity in contracts, 100k face) or 0 (quantity in units of 100 face)
df_scale = pd.read_sql_query(f"""
SELECT product_country, SUBSTR(reference_period, 1, 4) AS yr,
       CASE WHEN quantity IS NULL OR quantity = 0 OR price IS NULL OR price = 0 THEN NULL
            ELSE ROUND(LOG10(notional / (quantity * price))) END AS scale,
       COUNT(*) AS n_rows,
       COUNT(DISTINCT buyer_id) + COUNT(DISTINCT seller_id) AS n_leis,
       ROUND(SUM(notional) / 1e9, 1) AS notional_bn
FROM {schema}.hermesf_fut
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
""", cnxn)
df_scale.pivot_table(index=['product_country', 'yr'], columns='scale', values='notional_bn', fill_value=0)

In [ ]:
# diagnostics 2b: positions still reported after the contract expired
pd.read_sql_query(f"""
SELECT product_country, SUBSTR(reference_period, 1, 4) AS yr,
       COUNT(*) AS n_rows_after_expiry, ROUND(SUM(notional) / 1e9, 1) AS notional_bn
FROM {schema}.hermesf_fut
WHERE reference_period > expiration_date
GROUP BY 1, 2
ORDER BY 1, 2
""", cnxn)

In [ ]:
# diagnostics 3: the mid 2019 spike. Peak day per country, then the pairs driving it (cleaned notional)
def top_pairs_clean(country, day, n=15):
    return pd.read_sql_query(f"""
    SELECT buyer_id, buyer_sector, seller_id, seller_sector,
           COUNT(*) AS n_rows, COUNT(DISTINCT product_id) AS n_contracts,
           SUM(notional) / 1e9 AS notional_bn,
           ROUND(AVG(notional), 0) AS avg_notional, ROUND(AVG(notional_raw), 0) AS avg_notional_raw,
           ROUND(AVG(price_clean), 2) AS avg_price, ROUND(AVG(quantity), 0) AS avg_quantity
    FROM {schema}.hermesf_fut_clean
    WHERE product_country = '{country}' AND reference_period = '{day}'
    GROUP BY buyer_id, buyer_sector, seller_id, seller_sector
    ORDER BY ABS(SUM(notional)) DESC
    LIMIT {n}
    """, cnxn)

d19 = df_net_clean[df_net_clean['bdate'].between('2019-01-01', '2019-12-31')]
peaks = d19.loc[d19.groupby('product_country')['net_position_bn'].apply(lambda s: s.abs().idxmax())]
print(peaks)
top_pairs_clean('DE', peaks.set_index('product_country').loc['DE', 'bdate'])

In [ ]:
top_pairs_clean('FR', peaks.set_index('product_country').loc['FR', 'bdate'])

In [ ]:
# diagnostics 4: implied number of contracts per row (notional / (price x 1000)), on the cleaned table
# split by whether the CCP is one of the two sides, bucketed, with rows and notional per bucket
df_contracts = pd.read_sql_query(f"""
SELECT product_country,
       CASE WHEN buyer_sector = 'CCP' OR seller_sector = 'CCP' THEN 'CCP pair' ELSE 'other pair' END AS pair_type,
       CASE WHEN price_clean IS NULL THEN 'no price'
            WHEN notional / (price_clean * 1000) <  10000 THEN 'a: < 10k'
            WHEN notional / (price_clean * 1000) <  50000 THEN 'b: 10k-50k'
            WHEN notional / (price_clean * 1000) < 100000 THEN 'c: 50k-100k'
            WHEN notional / (price_clean * 1000) < 500000 THEN 'd: 100k-500k'
            ELSE 'e: > 500k' END AS contracts_bucket,
       COUNT(*) AS n_rows,
       ROUND(SUM(notional) / 1e9, 1) AS notional_bn,
       SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) AS n_rows_no_quantity
FROM {schema}.hermesf_fut_clean
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
""", cnxn)
df_contracts.pivot_table(index=['product_country', 'pair_type'], columns='contracts_bucket', values=['n_rows', 'notional_bn'], fill_value=0)